In [1]:
import torch
from brevitas.export import export_qonnx

# BED Model

In [5]:
models_folder = './step_by_step'
model_qnn_filename = './BED_AIMET_FPGA_classifier__best_mean_F1__BIPOLAR_Out__QONNX.onnx' 

# FINN Flow

## Load Model and View

In [6]:
from finn.util.visualization import showSrc, showInNetron
from qonnx.util.cleanup import cleanup as qonnx_cleanup

In [7]:
showInNetron(model_qnn_filename)

Serving './BED_AIMET_FPGA_classifier__best_mean_F1__BIPOLAR_Out__QONNX.onnx' at http://0.0.0.0:8083


## Clean

In [8]:
qonnx_clean_filename = models_folder + '/01_clean.onnx'
qonnx_cleanup(model_qnn_filename, out_file=qonnx_clean_filename)

In [9]:
showInNetron(qonnx_clean_filename)

Stopping http://0.0.0.0:8083
Serving './step_by_step/01_clean.onnx' at http://0.0.0.0:8083


## Convert to FINN

In [10]:
from qonnx.core.modelwrapper import ModelWrapper

In [11]:
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

In [12]:
model = ModelWrapper(qonnx_clean_filename)
model = model.transform(ConvertQONNXtoFINN())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveStaticGraphInputs())

In [13]:
finn_tidy = models_folder + '/02_finn_tidy.onnx'
model.save(finn_tidy)

In [14]:
showInNetron(finn_tidy)

Stopping http://0.0.0.0:8083
Serving './step_by_step/02_finn_tidy.onnx' at http://0.0.0.0:8083


## Preprocess

In [15]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from qonnx.transformation.infer_datatypes import InferDataTypes

In [16]:
model = ModelWrapper(finn_tidy)
global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)
# preprocessing: torchvision's ToTensor divides uint8 inputs by 255
totensor_pyt = ToTensor()
chkpt_preproc_name = models_folder + "/prepro_node.onnx"
export_qonnx(totensor_pyt, torch.randn(ishape), chkpt_preproc_name)
qonnx_cleanup(chkpt_preproc_name, out_file=chkpt_preproc_name)
pre_model = ModelWrapper(chkpt_preproc_name)
pre_model = pre_model.transform(ConvertQONNXtoFINN())

# join preprocessing and core model
model = model.transform(MergeONNXModels(pre_model))
# add input quantization annotation: UINT8 for all BNN-PYNQ models
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

/home/gmoreno/uav/finn/deps/qonnx/src/qonnx/transformation/merge_onnx_models.py:70: UserWarning: [MergeONNXModels] opsets for models to merge differ: 14 vs 17, output model will use opset 17
  warnings.warn(
/home/gmoreno/uav/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


### Tidy again

In [17]:
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

In [18]:
finn_prepro = models_folder + '/03_finn_prepro.onnx'
model.save(finn_prepro)

In [19]:
showInNetron(finn_prepro)

Stopping http://0.0.0.0:8083
Serving './step_by_step/03_finn_prepro.onnx' at http://0.0.0.0:8083


## Streamline

In [20]:
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul

from qonnx.transformation.change_datalayout import ChangeDataLayoutQuantAvgPool2d
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

from finn.transformation.streamline import Streamline
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC

In [21]:
model = ModelWrapper(finn_prepro)

In [22]:
model = model.transform(Streamline())

model = model.transform(LowerConvsToMatMul())
model = model.transform(ChangeDataLayoutQuantAvgPool2d())
model = model.transform(MakeMaxPoolNHWC())

model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())

model = model.transform(Streamline())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())

/home/gmoreno/uav/finn/src/finn/transformation/streamline/absorb.py:166: RuntimeWarning: divide by zero encountered in divide
  Tnew = T / A.reshape(-1, 1)


In [23]:
finn_streamline = models_folder + '/04_finn_streamline.onnx'
model.save(finn_streamline)

In [24]:
showInNetron(finn_streamline)

Stopping http://0.0.0.0:8083
Serving './step_by_step/04_finn_streamline.onnx' at http://0.0.0.0:8083


# To HW Layers

In [25]:
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten

from qonnx.custom_op.registry import getCustomOp

# Convert Last Bipolar Node to Binary and set output accordingly.

In [34]:
model = ModelWrapper(finn_streamline)

In [35]:
Multithreshold_node = model.get_nodes_by_op_type("MultiThreshold") 

In [36]:
for node in Multithreshold_node:
    node_inst = getCustomOp(node)
    if node_inst.get_nodeattr("out_dtype") == "BIPOLAR":
        node_inst.set_nodeattr("out_dtype", "BINARY")
        node_inst.set_nodeattr("out_scale", 1.0)
        node_inst.set_nodeattr("out_bias", 0.0)
        print(f'{node.name} converted from Bipolar to Binary\n{node}')

# global_out_name = model.graph.output[0].name
# model.set_tensor_datatype(global_out_name, DataType["BINARY"])

MultiThreshold_21 converted from Bipolar to Binary
input: "MatMul_20_out0"
input: "MultiThreshold_21_param0"
output: "global_out"
name: "MultiThreshold_21"
op_type: "MultiThreshold"
attribute {
  name: "out_dtype"
  s: "BINARY"
  type: STRING
}
attribute {
  name: "out_scale"
  f: 1.0
  type: FLOAT
}
attribute {
  name: "out_bias"
  f: 0.0
  type: FLOAT
}
domain: "qonnx.custom_op.general"



# Convert Nodes with Float32 annotation to Int32

In [37]:
for node in Multithreshold_node:
    if model.get_tensor_datatype(node.input[1]) == "FLOAT32":
        print(f'{node.name}: node with Float32 annotation')
        model.set_tensor_datatype(node.input[1], DataType["INT32"])
        print(f'{node.name}: changed to datatype {model.get_tensor_datatype(node.input[1])}')

In [38]:
global_out_name = model.graph.output[0].name
global_out_name

'global_out'

In [39]:
model.set_tensor_datatype(global_out_name, DataType["BINARY"])

In [40]:
finn_bipolar_to_binary = models_folder + '/05_finn_bipolar_to_binary.onnx'
model.save(finn_bipolar_to_binary)

In [41]:
showInNetron(finn_bipolar_to_binary)

Stopping http://0.0.0.0:8083
Serving './step_by_step/05_finn_bipolar_to_binary.onnx' at http://0.0.0.0:8083


### Standalone before MVAU conversion

In [42]:
model = ModelWrapper(finn_bipolar_to_binary)

In [43]:
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())

model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
model = model.transform(to_hw.InferPool())
model = model.transform(to_hw.InferConvInpGen())

model = model.transform(RemoveCNVtoFCFlatten())
model = model.transform(absorb.AbsorbConsecutiveTransposes())

model = model.transform(Streamline())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())

In [44]:
finn_hw_standalone = models_folder + '/06_finn_hw_standalone.onnx'
model.save(finn_hw_standalone)

In [45]:
showInNetron(finn_hw_standalone)

Stopping http://0.0.0.0:8083
Serving './step_by_step/06_finn_hw_standalone.onnx' at http://0.0.0.0:8083


# Dataflow Partition

In [68]:
model = ModelWrapper(finn_hw_standalone)
parent_model = model.transform(CreateDataflowPartition())

In [69]:
finn_parent_filename = models_folder + '/07_finn_dataflow_parent.onnx'
parent_model.save(finn_parent_filename)

In [70]:
showInNetron(finn_parent_filename)

Stopping http://0.0.0.0:8083
Serving './step_by_step/07_finn_dataflow_parent.onnx' at http://0.0.0.0:8083


In [71]:
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_filename = sdp_node.get_nodeattr("model")
dataflow_model = ModelWrapper(dataflow_filename)

In [72]:
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers

In [73]:
from finn.util.basic import pynq_part_map
# change this if you have a different PYNQ board, see list above
pynq_board = "Pynq-Z1"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10

# Specialize Layers

In [74]:
# FMPadding_node = dataflow_model.get_nodes_by_op_type("FMPadding")

# for node in FMPadding_node:
#     node_inst = getCustomOp(node)
#     node_inst.set_nodeattr("preferred_impl_style", "hls")
#     print(f'Node {node.name} forced to HLS')

In [75]:
# save the dataflow partition with a different name for easier access
# and specialize the layers to HLS variants
dataflow_model = dataflow_model.transform(SpecializeLayers(fpga_part))

dataflow_model = dataflow_model.transform(GiveUniqueNodeNames())
dataflow_model = dataflow_model.transform(GiveReadableTensorNames())

In [76]:
finn_dataflow_filename = models_folder + '/08_finn_dataflow_model.onnx'
dataflow_model.save(finn_dataflow_filename)

In [77]:
showInNetron(finn_dataflow_filename)

Stopping http://0.0.0.0:8083
Serving './step_by_step/08_finn_dataflow_model.onnx' at http://0.0.0.0:8083


# Folding Factors

In [78]:
from finn.transformation.fpgadataflow.set_folding import SetFolding

In [79]:
model = ModelWrapper(finn_dataflow_filename)

In [80]:
model = model.transform(SetFolding(
    target_cycles_per_frame=100000,
    mvau_wwidth_max=10000,
    two_pass_relaxation=True)
)

/home/gmoreno/uav/finn/src/finn/transformation/fpgadataflow/set_folding.py:221: UserWarning: SetFolding doesn't know how to handle op_type FMPadding_rtl
  warnings.warn("SetFolding doesn't know how to handle op_type " + op_type)
/home/gmoreno/uav/finn/src/finn/transformation/fpgadataflow/set_folding.py:233: UserWarning: Node ConvolutionInputGenerator_rtl_16 is bottleneck with 150437 cycles, running second pass
  warnings.warn(


In [81]:
folding_filename = models_folder + '/09_finn_folding.onnx'
model.save(folding_filename)

In [83]:
showInNetron(folding_filename)

Stopping http://0.0.0.0:8083
Serving './step_by_step/09_finn_folding.onnx' at http://0.0.0.0:8083


### Check estimated cycles per layer

In [66]:
import numpy as np

In [67]:
all_nodes = model.get_finn_nodes()

In [68]:
i = 0
total_cycles = []
for node in all_nodes:
    my_node = getCustomOp(node)
    node_cycles = my_node.get_nodeattr("cycles_estimate")
    total_cycles.append(node_cycles)
    print(f'Node {i} estimated cycles: {node_cycles}')
    i += 1
print(f'\nTotal estimated cycles: {np.array(total_cycles).sum()}')

Node 0 estimated cycles: 150528
Node 1 estimated cycles: 153228
Node 2 estimated cycles: 340053
Node 3 estimated cycles: 301056
Node 4 estimated cycles: 200704
Node 5 estimated cycles: 207936
Node 6 estimated cycles: 226254
Node 7 estimated cycles: 225792
Node 8 estimated cycles: 200704
Node 9 estimated cycles: 200704
Node 10 estimated cycles: 100352
Node 11 estimated cycles: 200704
Node 12 estimated cycles: 200704
Node 13 estimated cycles: 207936
Node 14 estimated cycles: 250504
Node 15 estimated cycles: 225792
Node 16 estimated cycles: 50176
Node 17 estimated cycles: 200704
Node 18 estimated cycles: 50176
Node 19 estimated cycles: 50176
Node 20 estimated cycles: 200704
Node 21 estimated cycles: 100352
Node 22 estimated cycles: 107648
Node 23 estimated cycles: 226744
Node 24 estimated cycles: 225792
Node 25 estimated cycles: 100352
Node 26 estimated cycles: 200704
Node 27 estimated cycles: 50176
Node 28 estimated cycles: 50176
Node 29 estimated cycles: 50176
Node 30 estimated cycles: 

# Minimize

In [69]:
from finn.transformation.fpgadataflow.minimize_accumulator_width import (
    MinimizeAccumulatorWidth,
)
from finn.transformation.fpgadataflow.minimize_weight_bit_width import (
    MinimizeWeightBitWidth,
)

In [70]:
model = ModelWrapper(folding_filename)

In [71]:
model = model.transform(MinimizeAccumulatorWidth())
model = model.transform(MinimizeWeightBitWidth())

/home/gmoreno/uav/finn/src/finn/custom_op/fpgadataflow/thresholding.py:84: UserWarning: inputDataType changing for Thresholding_rtl_1: INT32 -> INT15 
  warnings.warn(warn_str)
/home/gmoreno/uav/finn/src/finn/custom_op/fpgadataflow/thresholding.py:84: UserWarning: inputDataType changing for Thresholding_rtl_2: INT32 -> INT10 
  warnings.warn(warn_str)
/home/gmoreno/uav/finn/src/finn/custom_op/fpgadataflow/thresholding.py:84: UserWarning: inputDataType changing for Thresholding_rtl_3: INT32 -> INT12 
  warnings.warn(warn_str)
/home/gmoreno/uav/finn/src/finn/custom_op/fpgadataflow/thresholding.py:84: UserWarning: inputDataType changing for Thresholding_rtl_4: INT32 -> INT10 
  warnings.warn(warn_str)
/home/gmoreno/uav/finn/src/finn/custom_op/fpgadataflow/thresholding.py:84: UserWarning: inputDataType changing for Thresholding_rtl_5: INT32 -> INT11 
  warnings.warn(warn_str)
/home/gmoreno/uav/finn/src/finn/custom_op/fpgadataflow/thresholding.py:84: UserWarning: inputDataType changing for 

In [72]:
minimize_filename = models_folder + '/71_finn_minimize.onnx'
model.save(minimize_filename)

In [73]:
showInNetron(minimize_filename)

Stopping http://0.0.0.0:8083
Serving './step_by_step_onnx_trained_standalone_duplicateLayer/71_finn_minimize.onnx' at http://0.0.0.0:8083
